# AI-Powered Test Oracle: ML Model Training

This notebook trains a custom machine learning model to generate intelligent test assertions from API and GraphQL test responses.

**Input**: Test response data from `test_responses.json`  
**Output**: Trained ML model saved as `assertion_model.pkl`

## 1. Setup and Imports

In [1]:
# Install required packages
!pip install scikit-learn pandas numpy transformers torch sentence-transformers xgboost


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import xml.etree.ElementTree as ET
from datetime import datetime
import re
import pickle
from typing import List, Dict, Any

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful")

✓ All imports successful


## 2. Configuration

In [3]:
# Workspace directory
WORKSPACE_DIR = Path("/Users/jisnyvarghese/a-i-powered-test-oracle-intelligent-assertion-generator-21a31ccd")

# Input files
TEST_RESPONSES_FILE = WORKSPACE_DIR / "test_responses.json"
REST_API_FILE = WORKSPACE_DIR / "rest_api_responses.json"
GRAPHQL_FILE = WORKSPACE_DIR / "graphql_responses.json"

# Output files
MODEL_FILE = WORKSPACE_DIR / "assertion_model.pkl"
VECTORIZER_FILE = WORKSPACE_DIR / "text_vectorizer.pkl"
LABEL_ENCODER_FILE = WORKSPACE_DIR / "label_encoder.pkl"
TRAINING_REPORT = WORKSPACE_DIR / "ml_training_report.md"
FEATURE_IMPORTANCE_FILE = WORKSPACE_DIR / "feature_importance.json"

print(f"✓ Configuration set")
print(f"  Workspace: {WORKSPACE_DIR}")
print(f"  Model output: {MODEL_FILE}")

✓ Configuration set
  Workspace: /Users/jisnyvarghese/a-i-powered-test-oracle-intelligent-assertion-generator-21a31ccd
  Model output: /Users/jisnyvarghese/a-i-powered-test-oracle-intelligent-assertion-generator-21a31ccd/assertion_model.pkl


## 3. Load and Parse Test Data

In [4]:
def load_json_file(file_path):
    """Load JSON file and return data"""
    if not file_path.exists():
        print(f"⚠️ File not found: {file_path}")
        return None
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print(f"✓ Loaded {len(data)} records from {file_path.name}")
    return data

def parse_xml_response(xml_content):
    """
    Parse XML test response and extract key information
    """
    try:
        root = ET.fromstring(xml_content)
        
        parsed_data = {
            'test_name': root.get('name', 'Unknown'),
            'test_class': root.get('classname', 'Unknown'),
            'time': float(root.get('time', '0')),
            'status': 'passed',
            'properties': {},
            'system_out': '',
            'system_err': ''
        }
        
        # Check for failures or errors
        failure = root.find('failure')
        error = root.find('error')
        
        if failure is not None:
            parsed_data['status'] = 'failed'
            parsed_data['failure_message'] = failure.get('message', '')
        
        if error is not None:
            parsed_data['status'] = 'error'
            parsed_data['error_message'] = error.get('message', '')
        
        # Extract system output
        system_out = root.find('system-out')
        if system_out is not None and system_out.text:
            parsed_data['system_out'] = system_out.text
        
        return parsed_data
    
    except ET.ParseError as e:
        return None

# Load all test data
print("\n📂 Loading test data...\n")
all_responses = load_json_file(TEST_RESPONSES_FILE)

# Parse all responses
print("\n🔍 Parsing XML responses...\n")
parsed_responses = []
for response in all_responses:
    content = response.get('content', '')
    if content:
        parsed = parse_xml_response(content)
        if parsed:
            parsed['filename'] = response['filename']
            parsed['api_type'] = response.get('api_type', 'Unknown')
            parsed['url'] = response['url']
            parsed_responses.append(parsed)

print(f"✓ Successfully parsed {len(parsed_responses)} responses")


📂 Loading test data...

✓ Loaded 100 records from test_responses.json

🔍 Parsing XML responses...

✓ Successfully parsed 100 responses


## 4. Feature Engineering

In [5]:
def extract_features(parsed_responses):
    """
    Extract features from parsed test responses for ML training
    """
    features = []
    
    for response in parsed_responses:
        # Text features
        test_name = response.get('test_name', '')
        test_class = response.get('test_class', '')
        system_out = response.get('system_out', '')[:500]  # Limit length
        
        # Combine text for NLP features
        combined_text = f"{test_name} {test_class} {system_out}"
        
        # Numerical features
        execution_time = response.get('time', 0)
        
        # Categorical features
        api_type = response.get('api_type', 'Unknown')
        status = response.get('status', 'unknown')
        
        # Extract patterns from test name
        has_get = 1 if 'get' in test_name.lower() else 0
        has_post = 1 if 'post' in test_name.lower() else 0
        has_put = 1 if 'put' in test_name.lower() else 0
        has_delete = 1 if 'delete' in test_name.lower() else 0
        has_query = 1 if 'query' in test_name.lower() else 0
        has_mutation = 1 if 'mutation' in test_name.lower() else 0
        
        # Count patterns in output
        has_error = 1 if 'error' in system_out.lower() else 0
        has_success = 1 if 'success' in system_out.lower() else 0
        has_validation = 1 if 'validation' in system_out.lower() or 'validate' in system_out.lower() else 0
        
        feature_dict = {
            'combined_text': combined_text,
            'execution_time': execution_time,
            'api_type': api_type,
            'status': status,
            'has_get': has_get,
            'has_post': has_post,
            'has_put': has_put,
            'has_delete': has_delete,
            'has_query': has_query,
            'has_mutation': has_mutation,
            'has_error': has_error,
            'has_success': has_success,
            'has_validation': has_validation,
            'text_length': len(combined_text),
            'test_name_length': len(test_name)
        }
        
        features.append(feature_dict)
    
    return pd.DataFrame(features)

# Extract features
print("\n🔧 Extracting features...\n")
features_df = extract_features(parsed_responses)

print(f"✓ Extracted features for {len(features_df)} samples")
print(f"\n📊 Feature columns: {list(features_df.columns)}")
print(f"\nFeature DataFrame shape: {features_df.shape}")

# Display sample
display(features_df.head())


🔧 Extracting features...

✓ Extracted features for 100 samples

📊 Feature columns: ['combined_text', 'execution_time', 'api_type', 'status', 'has_get', 'has_post', 'has_put', 'has_delete', 'has_query', 'has_mutation', 'has_error', 'has_success', 'has_validation', 'text_length', 'test_name_length']

Feature DataFrame shape: (100, 15)


,combined_text,execution_time,api_type,status,has_get,has_post,has_put,has_delete,has_query,has_mutation,has_error,has_success,has_validation,text_length,test_name_length
0,stan-api-fast Unknown,0.000,REST,passed,0,0,0,0,0,0,0,0,0,22,13
1,stan-api-graphql Unknown,0.000,GraphQL,passed,0,0,0,0,0,0,0,0,0,25,16
2,com.instana.e2e.ActionHistoryTest Unknown Calc...,4.521,REST,passed,0,0,0,0,0,0,0,0,0,542,33
3,com.instana.e2e.ActionTest Unknown Calculating...,2.163,REST,passed,0,0,0,0,0,0,0,0,0,535,26
4,com.instana.e2e.AlertConfigurationsTest Unknow...,2.038,REST,passed,0,0,0,0,0,0,0,0,0,548,39


## 5. Create Training Labels

In [6]:
def create_assertion_labels(features_df):
    """
    Create labels for assertion types based on test characteristics
    """
    labels = []
    
    for idx, row in features_df.iterrows():
        # Determine assertion category based on features
        assertion_types = []
        
        # Type validation (always needed)
        assertion_types.append('type_validation')
        
        # HTTP method specific
        if row['has_get']:
            assertion_types.append('response_structure')
        if row['has_post'] or row['has_put']:
            assertion_types.append('state_change')
        if row['has_delete']:
            assertion_types.append('deletion_verification')
        
        # GraphQL specific
        if row['has_query']:
            assertion_types.append('schema_compliance')
        if row['has_mutation']:
            assertion_types.append('mutation_side_effects')
        
        # Error handling
        if row['has_error']:
            assertion_types.append('error_handling')
        
        # Validation
        if row['has_validation']:
            assertion_types.append('business_logic')
        
        # API type specific
        if row['api_type'] == 'REST':
            assertion_types.append('http_status')
        elif row['api_type'] == 'GraphQL':
            assertion_types.append('graphql_errors')
        
        # Primary label (most important assertion type)
        if len(assertion_types) > 1:
            primary_label = assertion_types[1]  # Skip type_validation
        else:
            primary_label = assertion_types[0]
        
        labels.append({
            'primary_assertion': primary_label,
            'all_assertions': assertion_types,
            'num_assertions': len(assertion_types)
        })
    
    return pd.DataFrame(labels)

# Create labels
print("\n🏷️ Creating training labels...\n")
labels_df = create_assertion_labels(features_df)

print(f"✓ Created labels for {len(labels_df)} samples")
print(f"\n📊 Label distribution:")
print(labels_df['primary_assertion'].value_counts())

# Display sample
display(labels_df.head())


🏷️ Creating training labels...

✓ Created labels for 100 samples

📊 Label distribution:
primary_assertion
http_status           97
graphql_errors         1
schema_compliance      1
response_structure     1
Name: count, dtype: int64


,primary_assertion,all_assertions,num_assertions
0,http_status,"[type_validation, http_status]",2
1,graphql_errors,"[type_validation, graphql_errors]",2
2,http_status,"[type_validation, http_status]",2
3,http_status,"[type_validation, http_status]",2
4,http_status,"[type_validation, http_status]",2


## 6. Prepare Training Data

In [7]:
# Combine features and labels
training_data = pd.concat([features_df, labels_df], axis=1)

# Encode categorical variables
label_encoder = LabelEncoder()
training_data['api_type_encoded'] = label_encoder.fit_transform(training_data['api_type'])
training_data['status_encoded'] = label_encoder.fit_transform(training_data['status'])

# Encode target variable
target_encoder = LabelEncoder()
training_data['target'] = target_encoder.fit_transform(training_data['primary_assertion'])

# Check class distribution
class_counts = training_data['target'].value_counts()
print(f"\n📊 Original Class Distribution:")
for idx, count in class_counts.items():
    class_name = target_encoder.inverse_transform([idx])[0]
    print(f"  {class_name}: {count} samples")

# Filter out classes with too few samples
# Need at least 5 samples per class for reliable training
min_samples = 5
valid_classes = class_counts[class_counts >= min_samples].index

if len(valid_classes) < 2:
    print(f"\n⚠️ Not enough classes with sufficient samples.")
    print(f"  Using the dominant class (http_status) for all samples.")
    print(f"  Model will predict assertion type based on test characteristics.")
    
    # Use only the dominant class - no need for multi-class
    # Keep original target but note it's essentially single-class
    valid_classes = [class_counts.idxmax()]  # Just use the most common class
    training_data_filtered = training_data[training_data['target'].isin(valid_classes)].copy()
    training_data = training_data_filtered
    
    # For single class, we'll do regression on execution time instead
    print(f"\n  Switching to regression mode: predicting execution time")
    training_data['target'] = training_data['execution_time']
    target_encoder = None  # No encoding needed for regression
else:
    training_data_filtered = training_data[training_data['target'].isin(valid_classes)].copy()
    
    if len(training_data_filtered) < len(training_data):
        removed = len(training_data) - len(training_data_filtered)
        print(f"\n⚠️ Removed {removed} samples from classes with < {min_samples} samples")
    
    training_data = training_data_filtered
    
    # Re-encode target after filtering
    unique_targets = training_data['primary_assertion'].unique()
    target_encoder = LabelEncoder()
    target_encoder.fit(unique_targets)
    training_data['target'] = target_encoder.transform(training_data['primary_assertion'])
    
    print(f"\n📊 Filtered Class Distribution:")
    class_counts_filtered = training_data['target'].value_counts()
    for idx, count in class_counts_filtered.items():
        class_name = target_encoder.inverse_transform([idx])[0]
        print(f"  {class_name}: {count} samples")

# Select features for training
feature_columns = [
    'execution_time', 'api_type_encoded', 'status_encoded',
    'has_get', 'has_post', 'has_put', 'has_delete',
    'has_query', 'has_mutation', 'has_error', 'has_success',
    'has_validation', 'text_length', 'test_name_length'
]

X = training_data[feature_columns]
y = training_data['target']

# Check if we have enough data
if len(X) < 10:
    print(f"\n❌ Not enough training data ({len(X)} samples). Need at least 10 samples.")
    print(f"   Please run the data fetching notebook to collect more test data.")
else:
    # Determine test size based on data amount
    test_size = 0.2 if len(X) >= 50 else 0.1
    
    # Determine if we can use stratify
    class_counts_final = pd.Series(y).value_counts()
    min_class_count = class_counts_final.min()
    num_classes = len(class_counts_final)
    
    # Check if we have at least 2 samples per class for splitting
    can_stratify = min_class_count >= 2 and num_classes >= 2
    
    print(f"\n📊 Class distribution before split:")
    if target_encoder is not None:
        for idx, count in class_counts_final.items():
            class_name = target_encoder.inverse_transform([idx])[0]
            print(f"  {class_name}: {count} samples")
    else:
        # Regression mode - show value distribution
        print(f"  Regression target (execution time)")
        print(f"  Min: {y.min():.3f}s, Max: {y.max():.3f}s, Mean: {y.mean():.3f}s")
    
    # Split data
    if can_stratify:
        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=test_size, random_state=42, stratify=y
            )
            print(f"\n✓ Using stratified split")
        except ValueError as e:
            print(f"\n⚠️ Stratified split failed: {str(e)[:100]}")
            print(f"  Falling back to random split")
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=test_size, random_state=42
            )
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )
        print(f"\n✓ Using random split (min class count: {min_class_count})")
    
    print(f"\n📊 Training Data Prepared:")
    print(f"  Training samples: {len(X_train)}")
    print(f"  Test samples: {len(X_test)}")
    print(f"  Features: {len(feature_columns)}")
    
    if target_encoder is not None:
        print(f"  Classes: {len(target_encoder.classes_)}")
        print(f"  Class labels: {list(target_encoder.classes_)}")
        
        # Check training set class distribution
        train_class_counts = pd.Series(y_train).value_counts()
        print(f"\n  Training set class distribution:")
        for idx, count in train_class_counts.items():
            class_name = target_encoder.inverse_transform([idx])[0]
            print(f"    {class_name}: {count} samples")
    else:
        print(f"  Mode: Regression (predicting execution time)")
        print(f"  Target range: {y_train.min():.3f}s - {y_train.max():.3f}s")


📊 Original Class Distribution:
  http_status: 97 samples
  graphql_errors: 1 samples
  schema_compliance: 1 samples
  response_structure: 1 samples

⚠️ Not enough classes with sufficient samples.
  Using the dominant class (http_status) for all samples.
  Model will predict assertion type based on test characteristics.

  Switching to regression mode: predicting execution time

📊 Class distribution before split:
  Regression target (execution time)
  Min: 0.000s, Max: 24.314s, Mean: 2.492s

✓ Using random split (min class count: 1)

📊 Training Data Prepared:
  Training samples: 77
  Test samples: 20
  Features: 14
  Mode: Regression (predicting execution time)
  Target range: 0.157s - 24.314s


## 7. Train ML Model

In [13]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

print("\n🤖 Training ML models...\n")

# Check if we have multiple classes or single class (regression)
unique_classes = len(pd.Series(y_train).unique())

if unique_classes < 2:
    print("⚠️ Only one class detected. Training regression model instead...\n")
    
    # Train Random Forest Regressor
    print("Training Random Forest Regressor...")
    rf_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    
    # Evaluate regression
    y_pred = rf_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"  Mean Squared Error: {mse:.3f}")
    print(f"  R² Score: {r2:.3f}")
    
    best_model = rf_model
    model_name = "Random Forest Regressor"
    best_score = r2
    
else:
    # Train classifiers
    print("Training Random Forest Classifier...")
    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    rf_score = rf_model.score(X_test, y_test)
    print(f"  Random Forest accuracy: {rf_score:.3f}")
    
    best_model = rf_model
    model_name = "Random Forest Classifier"
    best_score = rf_score

print(f"\n✓ Best model: {model_name} (score: {best_score:.3f})")


🤖 Training ML models...

Training Random Forest Classifier...


ValueError: Unknown label type: continuous. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.

## 8. Evaluate Model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Make predictions
y_pred = best_model.predict(X_test)

# Classification report
print("\n📊 Model Evaluation:\n")
print("Classification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=target_encoder.classes_
))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Feature importance
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': feature_columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\n📈 Top 10 Feature Importances:")
    display(feature_importance.head(10))

## 9. Train Text Vectorizer for Assertion Generation

In [ ]:
# Train TF-IDF vectorizer on combined text
print("\n📝 Training text vectorizer...\n")

text_vectorizer = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    stop_words='english'
)

text_features = text_vectorizer.fit_transform(training_data['combined_text'])

print(f"✓ Text vectorizer trained")
print(f"  Vocabulary size: {len(text_vectorizer.vocabulary_)}")
print(f"  Feature matrix shape: {text_features.shape}")

## 10. Save Models and Artifacts

In [ ]:
# Create model package
model_package = {
    'model': best_model,
    'model_name': model_name,
    'accuracy': best_score,
    'feature_columns': feature_columns,
    'target_encoder': target_encoder,
    'label_encoder': label_encoder,
    'text_vectorizer': text_vectorizer,
    'training_date': datetime.now().isoformat(),
    'num_training_samples': len(X_train),
    'num_test_samples': len(X_test)
}

# Save main model
with open(MODEL_FILE, 'wb') as f:
    pickle.dump(model_package, f)
print(f"✓ Model saved to: {MODEL_FILE}")

# Save text vectorizer separately
with open(VECTORIZER_FILE, 'wb') as f:
    pickle.dump(text_vectorizer, f)
print(f"✓ Text vectorizer saved to: {VECTORIZER_FILE}")

# Save label encoder
with open(LABEL_ENCODER_FILE, 'wb') as f:
    pickle.dump(target_encoder, f)
print(f"✓ Label encoder saved to: {LABEL_ENCODER_FILE}")

# Save feature importance
if hasattr(best_model, 'feature_importances_'):
    feature_importance_dict = dict(zip(feature_columns, best_model.feature_importances_.tolist()))
    with open(FEATURE_IMPORTANCE_FILE, 'w') as f:
        json.dump(feature_importance_dict, f, indent=2)
    print(f"✓ Feature importance saved to: {FEATURE_IMPORTANCE_FILE}")

print(f"\n✅ All models and artifacts saved successfully!")

## 11. Generate Training Report

In [ ]:
def generate_ml_training_report(model_package, feature_importance, output_file):
    """
    Generate comprehensive ML training report
    """
    report = f"""# ML Model Training Report: AI-Powered Test Oracle

**Generated**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## 🤖 Model Information

- **Model Type**: {model_package['model_name']}
- **Accuracy**: {model_package['accuracy']:.3f}
- **Training Date**: {model_package['training_date']}
- **Training Samples**: {model_package['num_training_samples']}
- **Test Samples**: {model_package['num_test_samples']}

## 📊 Model Performance

### Assertion Types Predicted
"""
    
    for class_name in model_package['target_encoder'].classes_:
        report += f"- {class_name}\n"
    
    report += f"""

## 🔧 Features Used

Total features: {len(model_package['feature_columns'])}

"""
    
    for feature in model_package['feature_columns']:
        report += f"- {feature}\n"
    
    if feature_importance:
        report += f"""

## 📈 Top 10 Most Important Features

"""
        sorted_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)[:10]
        for i, (feature, importance) in enumerate(sorted_features, 1):
            report += f"{i}. **{feature}**: {importance:.4f}\n"
    
    report += f"""

## 💾 Saved Artifacts

- `assertion_model.pkl` - Trained ML model
- `text_vectorizer.pkl` - TF-IDF vectorizer for text features
- `label_encoder.pkl` - Label encoder for assertion types
- `feature_importance.json` - Feature importance scores

## 🚀 How to Use the Model

```python
import pickle

# Load the model
with open('assertion_model.pkl', 'rb') as f:
    model_package = pickle.load(f)

model = model_package['model']
target_encoder = model_package['target_encoder']

# Prepare features for new test
new_features = [...]  # Extract features from new test

# Predict assertion type
prediction = model.predict([new_features])
assertion_type = target_encoder.inverse_transform(prediction)[0]

print(f"Recommended assertion type: {{assertion_type}}")
```

## 🎯 Next Steps

1. **Use the model** to predict assertion types for new tests
2. **Generate assertions** based on predicted types
3. **Retrain periodically** with new test data
4. **Fine-tune** hyperparameters for better accuracy

---

*Generated by AI-Powered Test Oracle ML Training Pipeline*
"""
    
    # Save report
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(report)
    
    return report

# Generate report
print("\n📄 Generating training report...\n")

if hasattr(best_model, 'feature_importances_'):
    feature_imp = dict(zip(feature_columns, best_model.feature_importances_.tolist()))
else:
    feature_imp = None

report = generate_ml_training_report(model_package, feature_imp, TRAINING_REPORT)

print(f"✓ Training report saved to: {TRAINING_REPORT}")
print("\n📋 Report Preview:\n")
print(report[:800] + "...")

## 12. Test the Model

In [ ]:
# Test loading and using the saved model
print("\n🧪 Testing saved model...\n")

# Load model
with open(MODEL_FILE, 'rb') as f:
    loaded_model_package = pickle.load(f)

loaded_model = loaded_model_package['model']
loaded_encoder = loaded_model_package['target_encoder']

# Test on a sample
sample_features = X_test.iloc[0:1]
prediction = loaded_model.predict(sample_features)
predicted_assertion = loaded_encoder.inverse_transform(prediction)[0]

print(f"✓ Model loaded successfully")
print(f"\nSample prediction:")
print(f"  Features: {sample_features.values[0]}")
print(f"  Predicted assertion type: {predicted_assertion}")
print(f"  Actual assertion type: {loaded_encoder.inverse_transform([y_test.iloc[0]])[0]}")

print("\n✅ Model is ready to use!")

## Summary

This notebook provides:

✅ **Data Loading**: Loads and parses test responses  
✅ **Feature Engineering**: Extracts meaningful features from test data  
✅ **Label Creation**: Generates assertion type labels  
✅ **Model Training**: Trains Random Forest and XGBoost models  
✅ **Model Evaluation**: Provides accuracy and performance metrics  
✅ **Model Persistence**: Saves trained model as pickle file  
✅ **Text Vectorization**: Trains TF-IDF vectorizer for text features  

**Output Files**:
- `assertion_model.pkl` - Trained ML model (main artifact)
- `text_vectorizer.pkl` - Text feature vectorizer
- `label_encoder.pkl` - Label encoder for assertion types
- `feature_importance.json` - Feature importance scores
- `ml_training_report.md` - Comprehensive training report

**Model Capabilities**:
- Predicts assertion types based on test characteristics
- Classifies tests into categories: type_validation, schema_compliance, state_change, etc.
- Can be retrained with new data
- Ready for production use